**Imports**

In [2]:
import os, re, json, csv
from lxml import etree as ET
import csv, json
import pandas as pd
import os, re, glob
import numpy as np
from collections import defaultdict

**This notebook contains code to extract relevant data from the metabolites xml file and match to the corresponding H-NMR spectra.**

**1. Build a list of HMDB ID's that have 1D HNMR data, based on filenames.**

Creates two files in processed_data:
- keep_ids_oned_h1.json : Contains HMDB ID's that are useable (have 1D H-NMR data)
- oned_h1_file_map.csv : Contains file names, HMDB ID's, and types

In [3]:
root = "original_data/hmdb_nmr_peak_lists"

id_re = re.compile(r'HMDB[\s_-]*(\d{5,7})', re.IGNORECASE)

def norm_id(d): return f"HMDB{int(d):07d}"

def looks_like_oned_h1(filepath):
    """
    Heuristics:
      1) filenames containing 'nmroned' -> accept
      2) otherwise, peek header/body:
         - reject if it mentions 'F1' AND 'F2' (2D)
         - accept if it contains '1H' and not '13C'/'15N' in axis/nucleus lines
         - quick numeric sniff: lines with two floats -> likely 2D; single float ppm -> likely 1D
    """
    fn = os.path.basename(filepath).lower()
    if "nmroned" in fn:
        return True
    if "nmrtwod" in fn:
        return False

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read(20000)
    except Exception:
        return False

    low = text.lower()
    if "f1" in low and "f2" in low:
        return False
    # nucleus hints
    if "13c" in low or "15n" in low:
        return False
    if "1h" in low or "proton" in low:
        pass  # keep checking

    # simple numeric sniff: count lines with 1 vs 2+ floats
    one_cols = two_plus_cols = 0
    for line in text.splitlines():
        toks = re.findall(r'[-+]?\d*\.\d+|\d+', line)
        if not toks: 
            continue
        # ignore very long lines (headers)
        if len(toks) == 1:
            one_cols += 1
        elif len(toks) >= 2:
            two_plus_cols += 1
        if one_cols + two_plus_cols > 50:
            break
    # If mostly single-column numbers -> likely 1D peak list
    return one_cols >= max(10, two_plus_cols * 2)

keep_ids = set()
rows = []
excluded = []
total_txt = 0

for dirpath, _, files in os.walk(root):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        total_txt += 1
        full = os.path.join(dirpath, fn)

        if not looks_like_oned_h1(full):
            excluded.append(os.path.relpath(full, root))
            continue

        # find HMDB id (filename first, then content)
        m = id_re.search(fn)
        digits = None
        if not m:
            try:
                with open(full, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read(20000)
                m = id_re.search(text)
            except Exception:
                m = None

        if m:
            digits = m.group(1)
            keep_ids.add(norm_id(digits))
            rows.append([os.path.relpath(full, root), norm_id(digits), "oned_h1"])
        else:
            excluded.append(os.path.relpath(full, root))

with open("processed_data/keep_ids_oned_h1.json", "w") as f:
    json.dump(sorted(keep_ids), f, indent=2)

with open("processed_data/oned_h1_file_map.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["file","hmdb_id","type"]); w.writerows(rows)

'''with open("excluded_non_oned_or_noid.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(excluded))'''

print("Total .txt files scanned:", total_txt)
print("Unique HMDB IDs with 1D 1H data:", len(keep_ids))
print("Example kept IDs:", list(sorted(keep_ids))[:10])



Total .txt files scanned: 1896
Unique HMDB IDs with 1D 1H data: 892
Example kept IDs: ['HMDB0000001', 'HMDB0000002', 'HMDB0000005', 'HMDB0000008', 'HMDB0000010', 'HMDB0000011', 'HMDB0000012', 'HMDB0000014', 'HMDB0000016', 'HMDB0000017']


**2. Match existing 1D H-NMR spectra files to relevant information in the metabolite xml file.**

Creates file in processed_data: 
- hmdb_subset_classes.csv : Contains HMDB ID's, name, kingdom, super_class, class and sub_class (these are the classifications that come with the dataset from HMDB)

In [4]:
xml_path = "original_data/hmdb_metabolites/hmdb_metabolites.xml"          # or .xml.gz if gzipped (use gzip.open)
ids_path = "processed_data/keep_ids_oned_h1.json"
out_csv  = "processed_data/hmdb_subset_classes.csv"

with open(ids_path) as f:
    keep_ids = set(json.load(f))

NS = "{http://www.hmdb.ca}"  # HMDB default namespace

def txt(parent, tag):
    """Safe text getter for direct child"""
    if parent is None: 
        return ""
    el = parent.find(NS + tag)
    return el.text.strip() if el is not None and el.text else ""

# Stream on end-of-element for <metabolite> to keep memory low
context = ET.iterparse(xml_path, events=("end",), tag=NS + "metabolite")

with open(out_csv, "w", newline="", encoding="utf-8") as fout:
    w = csv.writer(fout)
    w.writerow(["accession","name","kingdom","super_class","class","sub_class"])

    for _, metab in context:
        accession = txt(metab, "accession")
        if accession in keep_ids:
            name = txt(metab, "name")
            taxonomy = metab.find(NS + "taxonomy")
            row = [
                accession,
                name,
                txt(taxonomy, "kingdom"),
                txt(taxonomy, "super_class"),
                txt(taxonomy, "class"),
                txt(taxonomy, "sub_class"),   # <- the level you’ll use most
            ]
            w.writerow(row)

        # --- free memory ---
        metab.clear()
        # remove processed siblings from the tree to keep memory bounded
        while metab.getprevious() is not None:
            del metab.getparent()[0]

print(f"Done. Wrote classifications for {out_csv}")


Done. Wrote classifications for processed_data/hmdb_subset_classes.csv


**Explore super_class**

Creates a file in processed_data:
- super_class_counts.csv : Contains a list of the super_classes and counts for each.

In [5]:
# Load the file
df = pd.read_csv("processed_data/hmdb_subset_classes.csv")

# Count unique super_classes
unique_count = df["super_class"].nunique()
print(f"Number of unique super_classes: {unique_count}")

# Count compounds per super_class
counts = df["super_class"].value_counts()
print("\nCompounds per super_class:")
print(counts.to_string())

# Optionally, save to file
counts.to_csv("processed_data/super_class_counts.csv")


Number of unique super_classes: 11

Compounds per super_class:
super_class
Organic acids and derivatives              231
Lipids and lipid-like molecules            192
Organoheterocyclic compounds               134
Organic oxygen compounds                   105
Benzenoids                                  96
Nucleosides, nucleotides, and analogues     55
Phenylpropanoids and polyketides            39
Organic nitrogen compounds                  29
Alkaloids and derivatives                    4
Organosulfur compounds                       4
Hydrocarbons                                 1


**Create "other" group**

Creates a new column called group. The groups are the same as the super_classes, with the exception that rare super_classes are put into the group "other".

Creates files in processed_data:
- hmdb_subset_super_groups.csv : 
- group_counts.csv

Super classes with less entries than RARE_THRESHOLD are put in a group called "other"

In [6]:
# --- settings ---
in_csv   = "processed_data/hmdb_subset_classes.csv"
out_csv  = "processed_data/hmdb_subset_super_groups.csv"
counts_csv = "processed_data/group_counts.csv"
RARE_THRESHOLD = 10   # super_classes with < 10 entries → "Other"

# 1) Load
df = pd.read_csv(in_csv)

# 2) Build group from super_class
df["group"] = df["super_class"].fillna("Other")

# 3) Collapse rare groups into "Other"
super_counts = df["group"].value_counts()
rare_groups = set(super_counts[super_counts < RARE_THRESHOLD].index)
df.loc[df["group"].isin(rare_groups), "group"] = "Other"

# 4) Save outputs
df.to_csv(out_csv, index=False)
df["group"].value_counts().to_csv(counts_csv)

# 5) Print summary
total = len(df)
group_counts = df["group"].value_counts()
print("Groups and counts:\n", group_counts.to_string())
print(f"\nTotal compounds: {total}")
print(f"Unique groups: {df['group'].nunique()}")
print(f"Rare groups merged into 'Other' (threshold < {RARE_THRESHOLD}): {sorted(rare_groups)}")
print(f"\nWrote:\n - {out_csv}\n - {counts_csv}")


Groups and counts:
 group
Organic acids and derivatives              231
Lipids and lipid-like molecules            192
Organoheterocyclic compounds               134
Organic oxygen compounds                   105
Benzenoids                                  96
Nucleosides, nucleotides, and analogues     55
Phenylpropanoids and polyketides            39
Organic nitrogen compounds                  29
Other                                        9

Total compounds: 890
Unique groups: 9
Rare groups merged into 'Other' (threshold < 10): ['Alkaloids and derivatives', 'Hydrocarbons', 'Organosulfur compounds']

Wrote:
 - processed_data/hmdb_subset_super_groups.csv
 - processed_data/group_counts.csv


**Merge groups**

Super_classes have already been "converted" to groups, but these are currently identical except for the "other" group. Here we merge some groups together.

**IMPORTANT:** Overwrites hmdb_subset_super_groups.csv and group_counts.csv 

251010 - This cell now removes the "Other" group entirely from the dataset

In [7]:
# --- settings ---
in_csv      = "processed_data/hmdb_subset_classes.csv"
out_csv     = "processed_data/hmdb_subset_super_groups.csv"
counts_csv  = "processed_data/group_counts.csv"
RARE_THRESHOLD = 10   # after merging, any group with <10 → "Other"

# --- custom merges (exact super_class → merged group name) ---
MERGE_MAP = {
    # 1) Nitrogenous & organic acids
    "Organic acids and derivatives":        "Nitrogenous & organic acids",
    "Organic nitrogen compounds":           "Nitrogenous & organic acids",

    # 2) Lipids
    "Lipids and lipid-like molecules":      "Lipids",

    # 3) Aromatics
    "Benzenoids":                           "Aromatics",
    "Phenylpropanoids and polyketides":     "Aromatics",

    # 4) Heterocycles & nucleotides
    "Organoheterocyclic compounds":         "Heterocycles & nucleotides",
    "Nucleosides, nucleotides, and analogues": "Heterocycles & nucleotides",

    # 5) Organic oxygen compounds (kept as is, but named consistently)
    "Organic oxygen compounds":             "Organic oxygen compounds",
}

# 1) Load
df = pd.read_csv(in_csv)

# 2) Apply merges (default to original super_class if not in MERGE_MAP)
df["group"] = df["super_class"].map(MERGE_MAP).fillna(df["super_class"])
df["group"] = df["group"].fillna("Other")  # handle missing super_class

# 3) Collapse rare groups into "Other"
group_counts_before = df["group"].value_counts()
rare_groups = set(group_counts_before[group_counts_before < RARE_THRESHOLD].index)
df.loc[df["group"].isin(rare_groups), "group"] = "Other"

# ✅ 4) REMOVE "Other" entries entirely
df = df[df["group"] != "Other"]

# 5) Save outputs
df.to_csv(out_csv, index=False)
df["group"].value_counts().to_csv(counts_csv)

# 6) Print summary
total = len(df)
group_counts_after = df["group"].value_counts()
print("Groups and counts (after merging, excluding 'Other'):\n", group_counts_after.to_string())
print(f"\nTotal compounds: {total}")
print(f"Unique groups: {df['group'].nunique()}")
print(f"\nWrote:\n - {out_csv}\n - {counts_csv}")


Groups and counts (after merging, excluding 'Other'):
 group
Nitrogenous & organic acids    260
Lipids                         192
Heterocycles & nucleotides     189
Aromatics                      135
Organic oxygen compounds       105

Total compounds: 881
Unique groups: 5

Wrote:
 - processed_data/hmdb_subset_super_groups.csv
 - processed_data/group_counts.csv


**Create nmr_features_with_groups.csv**

In [8]:
# --- inputs ---
PEAK_DIR   = "original_data/hmdb_nmr_peak_lists"
GROUPS_CSV = "processed_data/hmdb_subset_super_groups.csv"   # accession, group

# --- binning ---
PPM_MIN, PPM_MAX, NBINS = -1.0, 14.0, 3000
bins = np.linspace(PPM_MIN, PPM_MAX, NBINS + 1)

# patterns
id_re = re.compile(r'HMDB[\s_-]*(\d{5,7})', re.IGNORECASE)
norm = lambda d: f"HMDB{int(d):07d}"

def is_oned_h1(filepath):
    """Return True for 1D 1H peaklists. Prefer filename 'nmroned';
    otherwise require a 'Table of Peaks' with (ppm) header and no F1/F2."""
    fn = os.path.basename(filepath).lower()
    if "nmroned" in fn:
        return True
    if "nmrtwod" in fn:
        return False
    # content sniff
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            head = f.read(4000).lower()
    except Exception:
        return False
    if "f1" in head and "f2" in head:
        return False
    has_table = ("table of peaks" in head) and ("(ppm" in head)
    # must look like 1H ppm range somewhere
    return has_table

def parse_peak_file_structured(path):
    """Parse the three sections (Peaks / Assignments / Multiplets)."""
    section = None
    ppm_list, inten_list = [], []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            low = line.lower()

            # section markers (order can vary)
            if "table of peaks" in low:
                section = "peaks"; continue
            if "table of assignments" in low:
                section = "assign"; continue
            if "table of multiplets" in low:
                section = "mult"; continue

            # skip headers
            if any(k in low for k in ["no.", "(ppm)", "(hz)", "height", "exp. shift", "shift1"]):
                continue

            cols = re.split(r'\s{2,}|\t+', line)

            if section == "peaks" and len(cols) >= 4:
                try:
                    ppm = float(cols[1].replace(',', '.'))
                    hgt = float(cols[3].replace(',', '.'))
                    if PPM_MIN <= ppm <= PPM_MAX:
                        ppm_list.append(ppm)
                        inten_list.append(hgt if hgt > 0 else 1.0)
                except:
                    pass
                continue

            if section == "assign" and len(cols) >= 3:
                try:
                    ppm = float(cols[2].replace(',', '.'))
                    if PPM_MIN <= ppm <= PPM_MAX:
                        ppm_list.append(ppm)
                        inten_list.append(1.0)
                except:
                    pass
                continue

            if section == "mult" and len(cols) >= 2:
                try:
                    ppm = float(cols[1].replace(',', '.'))
                    if PPM_MIN <= ppm <= PPM_MAX:
                        ppm_list.append(ppm)
                        inten_list.append(1.0)
                except:
                    pass
                continue

    return np.array(ppm_list, float), np.array(inten_list, float)

# --- config ---
NORMALIZE_MODE = "max"   # options: "sum", "max", or None

def peaks_to_vector(ppm, inten):
    vec = np.zeros(NBINS, float)
    if ppm.size == 0:
        return vec

    idx = np.digitize(ppm, bins) - 1
    mask = (idx >= 0) & (idx < NBINS)

    for b, v in zip(idx[mask], inten[mask]):
        if v > vec[b]:
            vec[b] = v

    # --- log transform to compress dynamic range ---
    vec = np.log1p(vec)   # log(1 + x), safe for zeros

    # --- normalization step ---
    if NORMALIZE_MODE == "sum":
        total = vec.sum()
        if total > 0:
            vec = vec / total
    elif NORMALIZE_MODE == "max":
        max_val = vec.max()
        if max_val > 0:
            vec = vec / max_val
    # if NORMALIZE_MODE is None → skip

    return vec



# -------- scan, parse and build features (1D only) --------
diag = {
    "files_scanned": 0,
    "files_candidate_oned": 0,
    "files_skipped_non_oned": 0,
    "files_no_hmdb_id": 0,
    "files_parsed_zero": 0,
    "files_parsed_nonzero": 0,
    "total_peaks": 0,
}
rows = []

for path in glob.glob(os.path.join(PEAK_DIR, "**/*.txt"), recursive=True):
    if not os.path.isfile(path): 
        continue
    diag["files_scanned"] += 1

    if not is_oned_h1(path):
        diag["files_skipped_non_oned"] += 1
        continue
    diag["files_candidate_oned"] += 1

    # HMDB accession
    m = id_re.search(os.path.basename(path))
    if not m:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            head = f.read(5000)
        m = id_re.search(head)
    if not m:
        diag["files_no_hmdb_id"] += 1
        continue
    hmdb = norm(m.group(1))

    # parse peaks
    ppm, inten = parse_peak_file_structured(path)
    diag["total_peaks"] += int(ppm.size)
    vec = peaks_to_vector(ppm, inten)

    if vec.sum() > 0:
        diag["files_parsed_nonzero"] += 1
        rows.append((hmdb, os.path.basename(path), vec))
    else:
        diag["files_parsed_zero"] += 1
        # skip adding zero vectors

# aggregate per HMDB (max pooling across files)
from collections import defaultdict
agg = defaultdict(list)
for hmdb, fname, vec in rows:
    agg[hmdb].append(vec)
features = {hmdb: np.max(np.vstack(vs), axis=0) for hmdb, vs in agg.items()}

feat_df = pd.DataFrame.from_dict(features, orient="index",
                                 columns=[f"bin_{i}" for i in range(NBINS)])
feat_df.index.name = "accession"
feat_df.reset_index(inplace=True)

# join with groups
groups = pd.read_csv(GROUPS_CSV, usecols=["accession","group"])
df = groups.merge(feat_df, on="accession", how="inner")
df.to_csv("processed_data/nmr_features_with_groups.csv", index=False)

# diagnostics
print("Scanned .txt files:", diag["files_scanned"])
print("Candidate 1D files:", diag["files_candidate_oned"])
print("Skipped (non-1D/2D-like):", diag["files_skipped_non_oned"])
print("Skipped (no HMDB id):", diag["files_no_hmdb_id"])
print("Parsed with non-zero vectors:", diag["files_parsed_nonzero"])
print("Parsed but zero vectors:", diag["files_parsed_zero"])
print("Total peaks parsed:", diag["total_peaks"])
print("Final feature table shape:", df.shape)


Scanned .txt files: 1896
Candidate 1D files: 1002
Skipped (non-1D/2D-like): 894
Skipped (no HMDB id): 0
Parsed with non-zero vectors: 857
Parsed but zero vectors: 145
Total peaks parsed: 26017
Final feature table shape: (846, 3002)


**Quick sanity check**

Verifies three key sanity points:

- Every spectrum has data (no all-zero rows).

- The number of peaks per sample is reasonable.

- The feature file and grouping file reference the same set of metabolites.

In [9]:
df = pd.read_csv("processed_data/nmr_features_with_groups.csv")
X = df.drop(columns=["accession","group"]).values

# How many nonzero bins per sample?
nnz = (X > 0).sum(axis=1)
print("Samples:", len(nnz))
print("Nonzero bins per sample (median, min, max):", np.median(nnz), nnz.min(), nnz.max())

# Any all-zero rows slipped in?
print("All-zero rows:", int((nnz == 0).sum()))

# Which grouped IDs are missing features?
grp = pd.read_csv("processed_data/hmdb_subset_super_groups.csv", usecols=["accession","group"])
missing = set(grp.accession) - set(df.accession)
print("Grouped but missing features:", len(missing))


Samples: 846
Nonzero bins per sample (median, min, max): 12.0 1 141
All-zero rows: 0
Grouped but missing features: 35


**Comment on sanity check**

Grouped but missing features: 35

Means 35 compounds appear in your hmdb_subset_super_groups.csv (they have class labels) but their spectra were not found or processed successfully into the nmr_features_with_groups.csv.

This can happen if:

- Their TXT peaklist files were missing, empty, or corrupted.

- The file parser skipped them (e.g. couldn’t read certain formatting).

- You filtered out spectra without peaks in your preprocessing step.

**Check where the missing features are**

In [10]:
missing_df = grp[grp.accession.isin(missing)]
print(missing_df["group"].value_counts())

group
Heterocycles & nucleotides     10
Nitrogenous & organic acids    10
Aromatics                       7
Organic oxygen compounds        5
Lipids                          3
Name: count, dtype: int64


**Finished**

Continue to baseline_ML_model_pipeline.ipynb